# Project 09 · Multi-Agent Support Desk (Reference solution)

**Module 09 · Multi-Agent Systems** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 6–8 hours |
| 📋 Prerequisites | Lessons 09.01–09.05 |
| 🔑 Keys | `GROQ_API_KEY` (LIVE mode, optional) |
| 💸 Cost (LIVE) | $0 on the free tier: one conversation ≈ 3–6k tokens of `gpt-oss-20b` per pattern; the default `LIVE_LIMIT = 5` runs 5 conversations × 2 patterns (≈ 40–80 calls, 30–60k tokens). `LIVE_LIMIT = 15` for the full acceptance run (≈ 3× that; spread it over time on the free tier) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |
| 📄 Spec | [project/README.md](../README.md) (also in [COURSE_OUTLINE.md](../../../../COURSE_OUTLINE.md#project-09--multi-agent-support-desk)) |

**Goal:** the Orbita Cloud support desk, built **twice** from the same specialists, then measured: **Pattern A** (handoffs: triage → billing / technical / account) and **Pattern B** (subagents: one coordinator calling the specialists as tools). Shared FAQ search, a billing manager who must approve refunds over $50, 15 test conversations, and an ADR with your recommendation.

This is the **reference solution**. Try the [starter notebook](../09-project-support-desk.ipynb) first; compare afterwards (same cells, same order).

## Requirements (from the spec)

1. **Pattern A (handoffs):** triage → billing / technical / account agents, each with its own tools and system prompt.
2. **Pattern B (subagents):** one coordinator calling the same specialists as tools.
3. **Shared RAG** over the product FAQ (`mastery.rag`); **HITL** before any refund over the threshold ($50).
4. **A 15-conversation test set**; compare A vs B on resolution correctness, model calls, tokens and latency.

**Acceptance criteria:** both patterns pass ≥ 12/15 conversations · refunds over the threshold always interrupt · a written comparison (ADR) with a recommendation.

**Modes:** with `GROQ_API_KEY` the notebook is **LIVE** (`gpt-oss-20b` for every agent; the FAQ is a Chroma index with local embeddings when `fastembed` is installed). Without it, **OFFLINE DEMO**: `mastery.agents.support.offline_model()` (rule-based, reads its role from the system prompt and its options from the tool names) and BM25 over the FAQ. The billing manager is a function (`manager_review`), so Restart & Run All always completes.

| TODO | Where | What |
|---|---|---|
| 1 | Step 3 | `build_pattern_a`: triage + 3 specialists as agent nodes, `handoff_tool` transfers, `handoff_graph` |
| 2 | Step 4 | `build_pattern_b`: the specialists wrapped with `subagent_tool` (named `billing_agent`, ...) under a coordinator |
| 3 | Step 5 | `manager_review`: the billing manager's policy (approve full-invoice refunds, reject partial ones with feedback) |
| 4 | Step 7 | `write_adr`: your ADR with the measured table and a **Recommendation** section |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langgraph>=1.2.14,<2" "langchain-groq>=1.1.3,<2" "langchain-chroma>=1.1.0,<2"

**You should now see** nothing, or *"Note: you may need to restart the kernel to use updated packages."*

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/09-multi-agent-systems/).
import importlib.util
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out.
import os
import warnings

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 09 LIVE FAQ index): no usage telemetry.
# Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# fastembed (Project 09 LIVE FAQ index) imports tqdm.auto; without ipywidgets it warns once. Progress bars still work.
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

## Step 1 · Mode and models

In [ ]:
# Imports and settings used by the whole notebook.
import json
import math

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

from mastery.agents import UsageMeter, handoff_graph, handoff_tool, subagent_tool
from mastery.agents.support import (COORDINATOR_PROMPT, DESCRIPTIONS, DOMAINS, REFUND_THRESHOLD, TRIAGE_PROMPT,
                                    SupportDesk, build_custom_desk, evaluate, faq_search, load_conversations,
                                    make_specialists, make_support_tools, offline_model)
from mastery.config import is_set, load_env

load_env()
LIVE = is_set("GROQ_API_KEY")
LIVE_LIMIT = 5            # conversations per pattern in LIVE mode; 15 for the full acceptance run
OUT_DIR = REPO_ROOT / "data" / "outputs" / "m09"
OUT_DIR.mkdir(parents=True, exist_ok=True)

if LIVE:
    from mastery.models import get_model

    def model_for(role):
        """One model per agent (all gpt-oss-20b here; try a stronger one for the coordinator)."""
        return get_model("fast", temperature=0, reasoning_effort="low")
else:
    def model_for(role):
        return offline_model(role)

print("Mode:", "LIVE (gpt-oss-20b)" if LIVE else "OFFLINE DEMO (rule-based support model)")
print("Refund approval threshold: $", REFUND_THRESHOLD)

**You should now see** `Mode: OFFLINE DEMO (rule-based support model)` (or `LIVE (gpt-oss-20b)` with a key) and the $50 threshold.

## Step 2 · The desk, the FAQ and the test set

In [ ]:
# Shared backend + shared FAQ search (one tool set, used by BOTH patterns).
desk = SupportDesk.load()
if LIVE:
    from mastery.rag import default_embeddings, embedding_label

    embeddings = default_embeddings()                       # fastembed if installed, else HashingEmbeddings
    search = faq_search("chroma", embeddings=embeddings)    # Chroma index in data/vectorstores/
    print("FAQ search: Chroma +", embedding_label(embeddings))
else:
    search = faq_search("bm25")
    print("FAQ search: BM25 (keyless)")
tools = make_support_tools(desk, search)
conversations = load_conversations()
RUN = conversations[:LIVE_LIMIT] if LIVE else conversations
THRESHOLD_PASS = math.ceil(0.8 * len(RUN))                  # 12 of 15

print({d: [t.name for t in ts] for d, ts in tools.items()})
print(len(conversations), "test conversations;", len(RUN), "run in this mode; pass mark:", THRESHOLD_PASS)
print("top FAQ hit for '429':", search("429 Too Many Requests")[0].metadata["source"])

**You should now see** the FAQ search type (BM25 offline; Chroma + an embedding label in LIVE), the 7 tools by domain, `15 test conversations; 15 run in this mode; pass mark: 12` (LIVE default: `5 run`, pass mark `4`), and `data/samples/support-desk/faq/technical.md` as the top hit for `429`.

The test set is [`data/samples/support-desk/conversations.jsonl`](../../../../data/samples/support-desk/conversations.jsonl): each line has the user `turns`, the manager `decision` when a refund needs one, and `expect` (tools that must run, number of pauses, text the final answer must contain). `score()` also checks that no refund over $50 was issued without the manager.

## Step 3 · Pattern A: handoffs (TODO 1)

In [ ]:
# Pattern A: handoffs (triage -> billing / technical / account), agents as graph nodes.
def build_pattern_a(model_for, tools, checkpointer):

    transfers = {d: handoff_tool(d, description=f"Transfer to the {d} specialist: {DESCRIPTIONS[d]}", carry="turn")
                 for d in DOMAINS}
    specialists = make_specialists(tools, handoffs=True)
    agents = {"triage": create_agent(model_for("triage"), tools=list(transfers.values()),
                                     system_prompt=TRIAGE_PROMPT, name="triage")}
    for d, spec in specialists.items():
        agents[d] = spec.build(model_for(d), extra_tools=[t for other, t in transfers.items() if other != d])
    return handoff_graph(agents, entry="triage", checkpointer=checkpointer)


pattern_a = build_pattern_a(model_for, tools, InMemorySaver())
if pattern_a is None:
    print("❌ Pattern A not built yet (TODO 1)")
else:
    print("Pattern A nodes:", [n for n in pattern_a.get_graph().nodes if not n.startswith("__")])

**You should now see** (solution) `Pattern A nodes: ['triage', 'billing', 'technical', 'account']`. Starter: `❌ Pattern A not built yet (TODO 1)`.

## Step 4 · Pattern B: subagents (TODO 2)

In [ ]:
# Pattern B: subagents (one coordinator, the specialists as tools).
def build_pattern_b(model_for, tools, checkpointer):

    specialists = make_specialists(tools)
    subtools = [subagent_tool(specialists[d].build(model_for(d)), name=f"{d}_agent",
                              description=f"The {d} specialist. {DESCRIPTIONS[d]}") for d in DOMAINS]
    return create_agent(model_for("coordinator"), tools=subtools, system_prompt=COORDINATOR_PROMPT,
                        name="coordinator", checkpointer=checkpointer)


pattern_b = build_pattern_b(model_for, tools, InMemorySaver())
if pattern_b is None:
    print("❌ Pattern B not built yet (TODO 2)")
else:
    print("Pattern B coordinator tools:", list(pattern_b.get_graph().nodes["tools"].data.tools_by_name))

**You should now see** (solution) `Pattern B coordinator tools: ['billing_agent', 'technical_agent', 'account_agent']`. Starter: `❌ Pattern B not built yet (TODO 2)`.

## Step 5 · The billing manager (TODO 3)

In [ ]:
# The human in the loop, as a function (so Restart & Run All completes). Called for every refund over the threshold.
MANAGER_LOG = []


def manager_review(payload):
    MANAGER_LOG.append({k: payload.get(k) for k in ("customer_id", "invoice_id", "amount")})

    customer = desk.customers.get(payload["customer_id"], {})
    invoice = next((i for i in customer.get("invoices", []) if i["id"] == payload["invoice_id"]), None)
    if invoice is not None and abs(payload["amount"] - invoice["amount"]) < 0.005:
        return "approve"
    return {"action": "reject", "feedback": "partial-month refunds are not offered"}


# Quick check on the two over-threshold conversations, for each pattern that exists.
by_id = {c["id"]: c for c in conversations}
for name, app in (("A", pattern_a), ("B", pattern_b)):
    if app is None:
        print(f"Pattern {name}: not built")
        continue
    for conv_id in ("S02", "S03"):
        desk.reset()
        config = {"configurable": {"thread_id": f"hitl-{name}-{conv_id}"}}
        out = app.invoke({"messages": [{"role": "user", "content": by_id[conv_id]["turns"][0]}]}, config)
        paused = "__interrupt__" in out
        if paused:
            out = app.invoke(Command(resume=manager_review(out["__interrupt__"][0].value)), config)
        print(f"Pattern {name} {conv_id}: paused={paused} | {out['messages'][-1].content[:90]}")

**You should now see** (solution) four lines, all `paused=True`: S02 `Refunded $199.00 for INV-2001 (approved by billing manager) ...` and S03 `Refund of $120.00 for INV-2001 was not approved by a billing manager (partial-month refunds are not offered).`, for A and for B. Starter: `Pattern A: not built`, `Pattern B: not built`; once TODO 1–2 are done but not TODO 3, S03 gets **approved** (the stub approves everything), which the acceptance checks catch.

## Step 6 · Compare A vs B on the test set

In [ ]:
# Run both patterns on the test set, conversation by conversation (fresh graph per pattern).
def run_pattern(build):
    app = build(model_for, tools, InMemorySaver())
    if app is None:
        return None
    rows = []
    for conv in RUN:
        meter = UsageMeter()
        report = evaluate(app, [conv], desk, meter=meter, reviewer=manager_review)
        row = report["rows"][0]
        rows.append({**{k: row[k] for k in ("id", "passed", "refund_policy")},
                     "interrupts": row["outcome"]["interrupts"], "answer": row["outcome"]["answer"], "calls": meter.model_calls, "tokens": meter.total_tokens,
                     "seconds": round(meter.seconds, 2), "agents": meter.agents(), "outcome": row["outcome"]})
    return {"rows": rows, "passed": sum(r["passed"] for r in rows), "total": len(rows),
            "calls": sum(r["calls"] for r in rows), "tokens": sum(r["tokens"] for r in rows),
            "seconds": round(sum(r["seconds"] for r in rows), 2)}


reports = {"A (handoffs)": run_pattern(build_pattern_a), "B (subagents)": run_pattern(build_pattern_b)}
summary = [{"pattern": name, "passed": f"{r['passed']}/{r['total']}", "model calls": r["calls"],
            "tokens": f"{r['tokens']:,}", "seconds": r["seconds"]}
           for name, r in reports.items() if r is not None]
if summary:
    table(summary, ["pattern", "passed", "model calls", "tokens", "seconds"])
else:
    print("❌ no pattern built yet (TODO 1 and 2)")

if all(reports.values()):
    print()
    per_conv = [{"id": a["id"], "A": "✅" if a["passed"] else "❌", "B": "✅" if b["passed"] else "❌",
                 "A calls": a["calls"], "B calls": b["calls"], "A pauses": a["interrupts"], "B pauses": b["interrupts"]}
                for a, b in zip(reports["A (handoffs)"]["rows"], reports["B (subagents)"]["rows"])]
    table(per_conv, ["id", "A", "B", "A calls", "B calls", "A pauses", "B pauses"])

saved = {name: None if r is None else {**{k: v for k, v in r.items() if k != "rows"},
                                       "rows": [{k: v for k, v in row.items() if k != "outcome"} for row in r["rows"]]}
         for name, r in reports.items()}
(OUT_DIR / "project09-report.json").write_text(json.dumps(saved, indent=2, default=str), encoding="utf-8")
print("\nsaved data/outputs/m09/project09-report.json")

**You should now see** (solution, offline) `A (handoffs)  15/15  53` calls (~24K tokens) and `B (subagents)  15/15  71` calls (~27K tokens), then one row per conversation: B costs 1–3 calls more on each (the coordinator's two calls per turn; 8 vs 5 on the repeat request S12), and both pause exactly once on S02 and S03. Seconds are tiny offline. Starter: `❌ no pattern built yet (TODO 1 and 2)`.

**LIVE**, the numbers vary per run: expect a few ❌ (a specialist that answers instead of transferring, a coordinator that drops the customer ID) and seconds that track model calls. Read the failing rows in `reports[...]["rows"]` before you change prompts.

## Step 7 · The ADR (TODO 4)

In [ ]:
# The ADR: your recommendation, backed by the table above.
def write_adr(reports, path):

    rows = "\n".join(f"| {name} | {r['passed']}/{r['total']} | {r['calls']} | {r['tokens']:,} | {r['seconds']} |"
                      for name, r in reports.items())
    a, b = reports["A (handoffs)"], reports["B (subagents)"]
    over = [r for rep in (a, b) for r in rep["rows"] if by_id[r["id"]]["expect"].get("interrupts", 0) > 0]
    hitl_line = ("Every refund over the threshold paused for the manager in both patterns." if all(r["interrupts"] >= 1 for r in over)
                 else "⚠️ Not every refund over the threshold paused: see the failing rows before shipping either pattern.")
    winner = "A (handoffs)" if (a["passed"], -a["calls"]) >= (b["passed"], -b["calls"]) else "B (subagents)"
    text = f"""# ADR-09: Orbita Cloud support desk architecture

**Status:** accepted · **Mode measured:** {"LIVE gpt-oss-20b" if LIVE else "offline rule-based model"} · **Conversations:** {a['total']}

## Context
Three domains (billing, technical, account), 7 tools, shared FAQ search, refunds over ${REFUND_THRESHOLD:.0f}
need a billing manager's approval. Test set: one-shot, multi-turn, domain switch and multi-domain conversations.

## Evidence
| Pattern | Passed | Model calls | Tokens | Seconds |
|---|---|---|---|---|
{rows}

{hitl_line}

## Recommendation
**{winner}** as the default desk: the best pass rate, then the fewest model calls
(A: {a['calls']} calls, B: {b['calls']} calls for the same conversations). Handoffs send repeat turns straight to
the active specialist; subagents route every turn through the coordinator twice. Choose B when one reviewer of every
answer matters more than cost (e.g. regulated replies), or when specialists are owned and deployed by different teams.
Route refunds through a custom workflow (Pattern C) so the money rules live in code.

## Consequences
- Cost scales with turns: A ≈ {a['calls'] / a['total']:.1f} calls per conversation, B ≈ {b['calls'] / b['total']:.1f}.
- A needs a clear transfer map (who may hand off to whom) and a recursion limit; B needs the coordinator to pass the
  customer ID in every task.
- Revisit when a domain passes ~10 tools, a new domain is added, or the live pass rate drops below 80%.

## Harness (Module 11)
The **technical** specialist: its hard cases are multi-step investigations (logs, firmware notes, several searches,
a written report). A deep agent with planning, a filesystem and its own subagents fits that better than one prompt.
"""
    path.write_text(text, encoding="utf-8")
    return text


adr_path = OUT_DIR / "project09-ADR.md"
adr_path.unlink(missing_ok=True)          # a stale ADR from an earlier run must not pass the checks
if all(reports.values()):
    adr_text = write_adr(reports, adr_path)
    print(adr_text[:900] if adr_text else "❌ ADR not written yet (TODO 4)")
else:
    adr_text = ""
    print("❌ build both patterns first (TODO 1 and 2)")

**You should now see** (solution) the start of the ADR: status, context, the evidence table with both patterns, and the recommendation (offline: **A (handoffs)**, 53 vs 71 calls). It's saved to `data/outputs/m09/project09-ADR.md`. Starter: `❌ build both patterns first (TODO 1 and 2)`, then, after TODO 1–2, `❌ ADR not written yet (TODO 4)`.

## Step 8 · Acceptance checks

In [ ]:
# Acceptance checks (from the spec). ✅ / ❌ per check.
A, B = reports["A (handoffs)"], reports["B (subagents)"]
HITL_IDS = [c["id"] for c in RUN if c["expect"].get("interrupts", 0) > 0]      # S02, S03 (if in this run)

checks = {
    "Pattern A builds (handoffs graph with triage + 3 specialists)": pattern_a is not None and
        {"triage", "billing", "technical", "account"} <= set(pattern_a.get_graph().nodes),
    "Pattern B builds (a coordinator calling specialist agents)": B is not None and
        "coordinator" in B["rows"][0]["agents"] and len(B["rows"][0]["agents"]) > 1,
    f"Pattern A passes >= {THRESHOLD_PASS}/{len(RUN)}": A is not None and A["passed"] >= THRESHOLD_PASS,
    f"Pattern B passes >= {THRESHOLD_PASS}/{len(RUN)}": B is not None and B["passed"] >= THRESHOLD_PASS,
    "Refunds over the threshold always interrupt (A and B)": A is not None and B is not None and all(
        r["interrupts"] >= 1 for rep in (A, B) for r in rep["rows"] if r["id"] in HITL_IDS),
    "No over-threshold refund issued without a manager (A and B)": all(
        r["refund_policy"] for rep in (A, B) if rep is not None for r in rep["rows"]),
    "Manager policy: full refunds approved, partial refunds rejected": A is not None and all(
        (r["id"] != "S02" or "refunded" in r["answer"].lower()) and (r["id"] != "S03" or not r["outcome"]["refunds"])
        for r in A["rows"]),
    "ADR written with a Recommendation section": adr_path.exists() and
        "## Recommendation" in adr_path.read_text(encoding="utf-8"),
    "ADR contains the measured numbers for both patterns": bool(adr_text) and A is not None and B is not None and
        str(A["calls"]) in adr_text and str(B["calls"]) in adr_text,
}
for name, ok in checks.items():
    print("✅" if ok else "❌", name)
print(f"\n{sum(checks.values())}/{len(checks)} acceptance checks passed")
_ = (OUT_DIR / "project09-acceptance.json").write_text(json.dumps(checks, indent=2), encoding="utf-8")

**You should now see** `9/9 acceptance checks passed` (solution). Starter: `1/9`: only `No over-threshold refund issued without a manager` passes from the start (nothing ran, so nothing was issued); every TODO turns more checks green. The result is saved to `data/outputs/m09/project09-acceptance.json`.

## Step 9 · Stretch goals

In [ ]:
# Stretch: Pattern C (custom refund workflow) + handoffs for everything else, on the same test set.
if all(reports.values()):
    writer = model_for("writer") if LIVE else None          # offline: the reply is the facts, unchanged
    pattern_c = build_custom_desk(model_for, tools, desk, checkpointer=InMemorySaver(), writer=writer)
    meter = UsageMeter()
    report_c = evaluate(pattern_c, RUN, desk, meter=meter, reviewer=manager_review)
    table(summary + [{"pattern": "C (workflow + handoffs)", "passed": f"{report_c['passed']}/{report_c['total']}",
                      "model calls": meter.model_calls, "tokens": f"{meter.total_tokens:,}",
                      "seconds": round(meter.seconds, 2)}],
          ["pattern", "passed", "model calls", "tokens", "seconds"])
else:
    print("⏭️ Build patterns A and B first.")

**You should now see** (solution, offline) a third row, `C (workflow + handoffs)  15/15  42` calls: refunds through the custom workflow cost one model call each. Starter: `⏭️ Build patterns A and B first.`

**More stretch ideas:**

- **Harness candidate:** in your ADR, name the specialist that would benefit from a full harness (planning, files, its own subagents). You rebuild it as a deep agent in Module 11.
- **Stronger coordinator:** in LIVE mode, give only the coordinator `get_model("reasoning")` (`gpt-oss-120b`) and re-run Step 6. Does B's pass rate change? At what token cost?
- **One-agent handoffs:** compare Pattern A with `mastery.agents.support.build_single_agent_desk` (09.03 Part 1) on the same run.
- **Your own conversations:** add 5 hard ones (typos, two refunds in one message, "the invoice from last month") to a copy of the JSONL and re-run.

In [ ]:
# Housekeeping: nothing to close (in-memory checkpointers). Chroma files (LIVE) stay in data/vectorstores/ (git-ignored).
print("Outputs in", OUT_DIR.relative_to(REPO_ROOT), "->", sorted(p.name for p in OUT_DIR.glob("project09-*")))

## Findings (write yours)

| | Pattern A (handoffs) | Pattern B (subagents) |
|---|---|---|
| passed (offline / LIVE) | | |
| model calls / tokens / seconds | | |
| failures and their cause | | |
| refunds over $50 interrupted | | |

- Where did each pattern fail LIVE, and was it a **context** problem (missing ID, lost tool result) or a **routing** problem?
- Which pattern would you ship, and what would make you switch?
- Log the decision in [`docs/decision-log.md`](../../../../docs/decision-log.md) and keep the ADR with your notes.

## ✅ Before you commit

- [ ] Restart & Run All passes (LIVE and without a key).
- [ ] All TODOs done; 9/9 acceptance checks; ADR and findings written.
- [ ] At least one LIVE run with `LIVE_LIMIT = 15` (or a note why not, e.g. free-tier limits).
- [ ] Outputs cleared (`jupyter nbconvert --clear-output --inplace <notebook>`); no keys in outputs; `data/outputs/` and `data/vectorstores/` not committed.

---
## ➡️ What's next

🎉 **Project 09 done:** you built the same support desk two ways, measured it and wrote the decision down.

**Next:** **[Module 10 · Testing, Evaluation & Observability](../../../../modules/10-testing-evaluation-observability/README.md)**: turn the measurements you made by hand here into a real quality workflow (unit tests with fake models, LangSmith datasets and experiments, LLM-as-judge and trajectory evaluators, online monitoring). Then Module 11 (Deep Agents) adds a harness on top of everything in this module (planning, a virtual filesystem, built-in subagents, context management), and you'll rebuild the specialist you named in your ADR as a deep agent. See [COURSE_OUTLINE.md → Module 10](../../../../COURSE_OUTLINE.md#module-10--testing-evaluation--observability); when it ships, start with its README's **Before you start this module** block.